In [ ]:
import pandas as pd
import xarray as xr
import numpy as np
import cftime

# Define input and output file names
input_csv = 'combined_snow_water.csv'
output_nc = 'snow_water_equivalent.nc'

def csv_to_netcdf_optimized(input_csv, output_nc):
    """
    Ultra-optimized conversion of large CSV to NetCDF
    Uses chunked processing to minimize memory usage
    """
    # Use more efficient reading strategy
    print("Starting data processing...")
    
    # Read CSV in chunks to manage memory
    chunk_size = 10_000_000  # Adjust based on your system's memory
    
    # Prepare lists to collect unique coordinates
    unique_x = set()
    unique_y = set()
    unique_times = set()
    
    # First pass: collect unique coordinates
    print("Collecting unique coordinates...")
    for chunk in pd.read_csv(input_csv, chunksize=chunk_size, 
                              dtype={
                                  'X': 'float32', 
                                  'Y': 'float32', 
                                  'Value': 'float32', 
                                  'Month': 'int32', 
                                  'Year': 'int32'
                              }):
        unique_x.update(chunk['X'].unique())
        unique_y.update(chunk['Y'].unique())
        
        # Use cftime for more flexible time handling
        unique_times.update(
            cftime.Datetime360Day(row['Year'], row['Month'], 1) 
            for _, row in chunk.iterrows()
        )
    
    # Convert to sorted lists
    x_coords = sorted(list(unique_x))
    y_coords = sorted(list(unique_y))
    time_coords = sorted(list(unique_times))
    
    # Prepare 3D array for data
    print("Preparing data array...")
    data_3d = np.full((len(time_coords), len(y_coords), len(x_coords)), 
                      np.nan, dtype=np.float32)
    
    # Create mapping dictionaries for efficient indexing
    x_map = {x: i for i, x in enumerate(x_coords)}
    y_map = {y: i for i, y in enumerate(y_coords)}
    time_map = {t: i for i, t in enumerate(time_coords)}
    
    # Second pass: fill data array
    print("Filling data array...")
    for chunk in pd.read_csv(input_csv, chunksize=chunk_size, 
                              dtype={
                                  'X': 'float32', 
                                  'Y': 'float32', 
                                  'Value': 'float32', 
                                  'Month': 'int32', 
                                  'Year': 'int32'
                              }):
        # Fill data array
        for _, row in chunk.iterrows():
            # Create cftime coordinate
            time_coord = cftime.Datetime360Day(row['Year'], row['Month'], 1)
            
            t_idx = time_map[time_coord]
            y_idx = y_map[row['Y']]
            x_idx = x_map[row['X']]
            data_3d[t_idx, y_idx, x_idx] = row['Value']
    
    # Create xarray Dataset
    print("Creating xarray Dataset...")
    ds = xr.Dataset(
        data_vars={
            'snow_water_equivalent': (
                ['time', 'Y', 'X'], 
                data_3d,
                {
                    'units': 'mm',
                    'long_name': 'Snow Water Equivalent'
                }
            )
        },
        coords={
            'time': time_coords,
            'Y': y_coords,
            'X': x_coords
        }
    )
    
    # Set time encoding to handle cftime
    encoding = {
        'time': {
            'calendar': '360_day',
            'units': 'days since 1900-01-01'
        },
        'snow_water_equivalent': {
            'compression': 'zlib',
            'compression_level': 5,
            'chunksizes': (1, len(y_coords), len(x_coords))
        }
    }
    
    # Save to NetCDF with optimizations
    print(f"Saving to {output_nc}...")
    ds.to_netcdf(
        output_nc, 
        engine='netcdf4', 
        encoding=encoding
    )
    
    print(f"Conversion complete. Saved to {output_nc}")
    print(ds)

# Run the conversion
csv_to_netcdf_optimized(input_csv, output_nc)

Starting data processing...
